# Playground Series - S6E8: Predicting Smartphone Addiction

Binary classification task &mdash; predict the probability of `addicted_label` for each `id` in the test set.

**Evaluation metric:** ROC-AUC (Area Under the ROC Curve) between predicted probability and observed target.

Workflow:
1. Get the data (Kaggle / Kaggle-Colab / local) Or Download and Call the data in collab
2. Exploratory Data Analysis (EDA)
3. Preprocessing pipeline (auto-detects numeric vs. categorical columns, so it works regardless of the exact schema)
4. Model comparison via cross-validated ROC-AUC
5. Hyperparameter tuning of the best model
6. Final fit & probability predictions
7. Submission file generation (`id, addicted_label`)

> **About getting the data:** this is a live, synthetically-generated Playground competition, so the data lives only on Kaggle (`playground-series-s6e8`) &mdash; there's no public mirror to download it from elsewhere the way there is for the classic Titanic dataset. Cell 2 below handles Kaggle, Kaggle-in-Colab, and local runs automatically. If you're on Colab, see the instructions in that cell for a one-time Kaggle API setup (about 30 seconds).

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, RocCurveDisplay, classification_report

import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
pd.set_option('display.max_columns', None)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

TARGET = 'addicted_label'
ID_COL = 'id'

## 1. Get the Data

**On Kaggle Notebooks:** just add the `playground-series-s6e8` dataset via "Add Input" &mdash; the cell below finds it automatically at `/kaggle/input/playground-series-s6e8/`.

**On Colab (or anywhere else):** you need a free Kaggle account + API token, one time only:
1. Go to https://www.kaggle.com/settings &rarr; "Create New Token" &rarr; this downloads `kaggle.json`
2. Run the cell below &mdash; it will prompt you to upload that `kaggle.json` file, then it downloads the competition data for you automatically (you must have clicked "Join Competition" on the competition page first, since Kaggle requires accepting the rules before the API can fetch the data).

In [ ]:
COMPETITION = 'playground-series-s6e8'
KAGGLE_PATH = f'/kaggle/input/{COMPETITION}'

if os.path.exists(os.path.join(KAGGLE_PATH, 'train.csv')):
    # Running inside a Kaggle Notebook with the competition data attached
    DATA_DIR = KAGGLE_PATH
    print('Loading data from Kaggle input directory...')

elif os.path.exists('train.csv') and os.path.exists('test.csv'):
    # Files already downloaded into the working directory
    DATA_DIR = './'
    print('Loading data from local working directory...')

else:
    # Colab / other environments: fetch via the Kaggle API
    print('Data not found locally - fetching via the Kaggle API...')
    os.system('pip install -q kaggle')

    kaggle_json = os.path.expanduser('~/.kaggle/kaggle.json')
    if not os.path.exists(kaggle_json):
        try:
            # Colab file upload prompt
            from google.colab import files
            print('Please upload your kaggle.json (from https://www.kaggle.com/settings):')
            uploaded = files.upload()
            os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
            for fname in uploaded:
                os.rename(fname, kaggle_json)
            os.chmod(kaggle_json, 0o600)
        except ImportError:
            raise RuntimeError(
                'kaggle.json not found. Place it at ~/.kaggle/kaggle.json '
                '(get it from https://www.kaggle.com/settings) and re-run this cell.'
            )

    os.makedirs('competition_data', exist_ok=True)
    exit_code = os.system(
        f'kaggle competitions download -c {COMPETITION} -p competition_data'
    )
    if exit_code != 0:
        raise RuntimeError(
            f'Kaggle download failed. Make sure you have joined the "{COMPETITION}" '
            'competition (accepted its rules) at https://www.kaggle.com/competitions/'
            f'{COMPETITION}/rules with the same account as your API token.'
        )

    import zipfile
    zip_path = f'competition_data/{COMPETITION}.zip'
    if os.path.exists(zip_path):
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall('competition_data')

    DATA_DIR = 'competition_data/'
    print('Download complete.')

train_df = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test_df = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))

print(f'Train shape: {train_df.shape}')
print(f'Test shape:  {test_df.shape}')
train_df.head()

In [ ]:
train_df.info()

In [ ]:
train_df.describe(include='all').T

In [ ]:
# Missing values overview
missing = pd.DataFrame({
    'train_missing': train_df.isnull().sum(),
    'train_pct': (train_df.isnull().sum() / len(train_df) * 100).round(2),
})
missing = missing[missing['train_missing'] > 0].sort_values('train_missing', ascending=False)
print('Columns with missing values:' if len(missing) else 'No missing values in train set.')
missing

## 2. Exploratory Data Analysis

In [ ]:
print(f'Positive class rate ({TARGET}=1): {train_df[TARGET].mean():.2%}')

plt.figure(figsize=(5, 4))
sns.countplot(data=train_df, x=TARGET)
plt.title('Target Class Balance')
plt.show()

In [ ]:
# Split feature columns into numeric / categorical automatically, since the exact
# schema of this synthetic dataset can vary.
feature_cols = [c for c in train_df.columns if c not in [ID_COL, TARGET]]

numeric_features = [c for c in feature_cols if pd.api.types.is_numeric_dtype(train_df[c])]
categorical_features = [c for c in feature_cols if c not in numeric_features]

print(f'{len(numeric_features)} numeric features: {numeric_features}')
print(f'{len(categorical_features)} categorical features: {categorical_features}')

In [ ]:
 # Distributions of numeric features by target class (first 12 to keep it readable)
cols_to_plot = numeric_features[:12]
n_cols = 3
n_rows = int(np.ceil(len(cols_to_plot) / n_cols))

if cols_to_plot:
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4 * n_rows))
    axes = np.array(axes).reshape(-1)
    for i, col in enumerate(cols_to_plot):
        sns.kdeplot(data=train_df, x=col, hue=TARGET, ax=axes[i], common_norm=False, fill=True, alpha=0.3)
        axes[i].set_title(col)
    for j in range(len(cols_to_plot), len(axes)):
        axes[j].axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('No numeric features to plot.')

In [ ]:
# Target rate by categorical features (first 8)
cat_cols_to_plot = categorical_features[:8]

if cat_cols_to_plot:
    n_cols = 2
    n_rows = int(np.ceil(len(cat_cols_to_plot) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 4 * n_rows))
    axes = np.array(axes).reshape(-1)
    for i, col in enumerate(cat_cols_to_plot):
        order = train_df.groupby(col)[TARGET].mean().sort_values(ascending=False).index
        sns.barplot(data=train_df, x=col, y=TARGET, order=order, ax=axes[i])
        axes[i].set_title(f'{TARGET} rate by {col}')
        axes[i].tick_params(axis='x', rotation=45)
    for j in range(len(cat_cols_to_plot), len(axes)):
        axes[j].axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('No categorical features to plot.')

In [ ]:
if len(numeric_features) > 1:
    plt.figure(figsize=(min(1 + 0.6 * len(numeric_features), 16), min(1 + 0.6 * len(numeric_features), 14)))
    corr = train_df[numeric_features + [TARGET]].corr()
    sns.heatmap(corr, cmap='coolwarm', center=0, annot=len(numeric_features) <= 15, fmt='.2f')
    plt.title('Correlation Matrix')
    plt.show()

    print('\nTop features correlated with the target:')
    print(corr[TARGET].drop(TARGET).abs().sort_values(ascending=False).head(15))

## 3. Preprocessing Pipeline

In [ ]:
X = train_df[feature_cols].copy()
y = train_df[TARGET].copy()
X_test = test_df[feature_cols].copy()

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

## 4. Model Comparison (Cross-Validated ROC-AUC)

In [ ]:
import time
import warnings
warnings.filterwarnings("ignore")

from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier
)

# -----------------------------
# Remove rows with NaN target
# -----------------------------
nan_mask = y.isna()

if nan_mask.any():
    print(f"Removing {nan_mask.sum()} rows with NaN target.")
    X_filtered = X.loc[~nan_mask].copy()
    y_filtered = y.loc[~nan_mask].copy()
else:
    X_filtered = X.copy()
    y_filtered = y.copy()

print("=" * 60)
print("Dataset Shape")
print("X:", X_filtered.shape)
print("y:", y_filtered.shape)
print("=" * 60)

# -----------------------------
# Models
# -----------------------------
models = {
    "LogisticRegression": LogisticRegression(
        max_iter=2000,
        random_state=RANDOM_STATE
    ),

    "RandomForest": RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),

    "GradientBoosting": GradientBoostingClassifier(
        random_state=RANDOM_STATE
    ),

    "HistGradientBoosting": HistGradientBoostingClassifier(
        random_state=RANDOM_STATE
    )
}

# Optional LightGBM
try:
    from lightgbm import LGBMClassifier

    models["LightGBM"] = LGBMClassifier(
        n_estimators=100,
        random_state=RANDOM_STATE,
        verbosity=-1
    )
except ImportError:
    print("LightGBM not installed.")

# Optional XGBoost
try:
    from xgboost import XGBClassifier

    models["XGBoost"] = XGBClassifier(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        tree_method="hist",
        eval_metric="auc",
        random_state=RANDOM_STATE,
        verbosity=0
    )
except ImportError:
    print("XGBoost not installed.")

# -----------------------------
# Cross Validation
# -----------------------------
cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=RANDOM_STATE
)

results = {}

print("\nStarting model evaluation...\n")

for name, model in models.items():

    print("-" * 60)
    print(f"Training: {name}")

    start = time.time()

    try:
        pipe = Pipeline([
            ("preprocessor", preprocessor),
            ("classifier", model)
        ])

        scores = cross_val_score(
            pipe,
            X_filtered,
            y_filtered,
            scoring="roc_auc",
            cv=cv,
            n_jobs=1
        )

        elapsed = time.time() - start

        results[name] = scores

        print(f"Mean AUC : {scores.mean():.5f}")
        print(f"Std AUC  : {scores.std():.5f}")
        print(f"Time     : {elapsed:.2f} sec")

    except Exception as e:
        print(f"{name} failed.")
        print(e)

print("\n" + "=" * 60)
print("Summary")
print("=" * 60)

for name, scores in results.items():
    print(f"{name:<25} AUC={scores.mean():.5f}")

In [ ]:
results_df = pd.DataFrame(results)
plt.figure(figsize=(10, 5))
sns.boxplot(data=results_df)
plt.ylabel('CV ROC-AUC')
plt.title('Model Comparison (5-fold Stratified CV)')
plt.xticks(rotation=15)
plt.show()

results_df.mean().sort_values(ascending=False)

## 5. Hyperparameter Tuning

Tunes the strongest model from the comparison above. Edit the relevant grid if you want a wider/narrower search.

In [ ]:
best_model_name = results_df.mean().idxmax()
print(f'Best model from CV: {best_model_name}')

In [ ]:
import time
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold

# ----------------------------
# Remove NaN targets
# ----------------------------
nan_mask = y.isna()

if nan_mask.any():
    print(f"Removing {nan_mask.sum()} rows with NaN target.")
    X_filtered = X.loc[~nan_mask].copy()
    y_filtered = y.loc[~nan_mask].copy()
else:
    X_filtered = X.copy()
    y_filtered = y.copy()

print(f"Dataset Shape: {X_filtered.shape}")

# ----------------------------
# Smaller parameter grids
# ----------------------------
param_grids = {

    'RandomForest': {
        'classifier__n_estimators': [200, 400, 600],
        'classifier__max_depth': [8, 12, None],
        'classifier__min_samples_split': [2, 5],
        'classifier__min_samples_leaf': [1, 2],
    },

    'GradientBoosting': {
        'classifier__n_estimators': [100, 200, 300],
        'classifier__learning_rate': [0.03, 0.05, 0.1],
        'classifier__max_depth': [2, 3],
    },

    'HistGradientBoosting': {
        'classifier__max_iter': [150, 250, 350],
        'classifier__learning_rate': [0.03, 0.05, 0.1],
        'classifier__max_depth': [None, 4, 6],
        'classifier__l2_regularization': [0.0, 0.1],
    },

    'LogisticRegression': {
        'classifier__C': [0.01, 0.1, 1, 10],
    },

    'LightGBM': {
        'classifier__n_estimators': [200, 400, 600],
        'classifier__learning_rate': [0.03, 0.05, 0.1],
        'classifier__num_leaves': [31, 63],
        'classifier__max_depth': [-1, 6, 10],
    },

    'XGBoost': {
        'classifier__n_estimators': [200, 400, 600],
        'classifier__learning_rate': [0.03, 0.05, 0.1],
        'classifier__max_depth': [3, 5, 7],
        'classifier__subsample': [0.8, 1.0],
    },
}

# ----------------------------
# Cross Validation
# ----------------------------
cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=RANDOM_STATE
)

# ----------------------------
# Build Pipeline
# ----------------------------
best_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', models[best_model_name])
])

print("=" * 60)
print("Best Model:", best_model_name)
print("Starting Hyperparameter Search...")
print("=" * 60)

start = time.time()

search = RandomizedSearchCV(
    estimator=best_pipe,
    param_distributions=param_grids[best_model_name],
    n_iter=20,                 # Increase to 30-50 if you have more time
    scoring='roc_auc',
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=1,                  # Prevent freezing
    verbose=2,
    return_train_score=True
)

search.fit(X_filtered, y_filtered)

elapsed = time.time() - start

print("\n" + "=" * 60)
print("Hyperparameter Search Complete")
print("=" * 60)

print(f"Time Taken : {elapsed / 60:.2f} minutes")
print(f"Best ROC-AUC : {search.best_score_:.6f}")
print("\nBest Parameters:")

for k, v in search.best_params_.items():
    print(f"{k}: {v}")

best_model = search.best_estimator_

print("\nBest Pipeline:")
print(best_model)

NameError: name 'y' is not defined

## 6. Validation Check

A holdout split just to visualize an ROC curve alongside the CV score above.

In [ ]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

val_model = grid_search.best_estimator_
val_model.fit(X_tr, y_tr)
val_probs = val_model.predict_proba(X_val)[:, 1]

print(f'Holdout ROC-AUC: {roc_auc_score(y_val, val_probs):.5f}\n')
print(classification_report(y_val, (val_probs >= 0.5).astype(int)))

plt.figure(figsize=(6, 6))
RocCurveDisplay.from_predictions(y_val, val_probs)
plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
plt.title('ROC Curve (Holdout Set)')
plt.show()

## 7. Feature Importance (if supported by the chosen model)

In [ ]:
clf = val_model.named_steps['classifier']

if hasattr(clf, 'feature_importances_'):
    feature_names = val_model.named_steps['preprocessor'].get_feature_names_out()
    importances = pd.Series(clf.feature_importances_, index=feature_names).sort_values(ascending=False)

    plt.figure(figsize=(8, 8))
    importances.head(20).sort_values().plot(kind='barh')
    plt.title(f'Top 20 Feature Importances ({best_model_name})')
    plt.tight_layout()
    plt.show()
elif hasattr(clf, 'coef_'):
    feature_names = val_model.named_steps['preprocessor'].get_feature_names_out()
    coefs = pd.Series(clf.coef_[0], index=feature_names).sort_values(key=np.abs, ascending=False)

    plt.figure(figsize=(8, 8))
    coefs.head(20).sort_values().plot(kind='barh')
    plt.title(f'Top 20 Coefficients (abs value) ({best_model_name})')
    plt.tight_layout()
    plt.show()
else:
    print(f'{best_model_name} does not expose feature_importances_ or coef_.')

## 8. Final Fit on Full Training Data & Predict Probabilities on Test Set

In [ ]:
# Refit the tuned pipeline on ALL available training data before predicting on test
final_model = grid_search.best_estimator_
final_model.fit(X, y)

test_probs = final_model.predict_proba(X_test)[:, 1]

print(f'Predicted probability summary:\n{pd.Series(test_probs).describe()}')

## 9. Create Submission File

In [ ]:
submission = pd.DataFrame({
    ID_COL: test_df[ID_COL],
    TARGET: test_probs
})

assert submission.shape[0] == test_df.shape[0], 'Submission row count must match test set'
assert list(submission.columns) == [ID_COL, TARGET], f'Columns must be {ID_COL}, {TARGET}'
assert submission[TARGET].between(0, 1).all(), 'Predicted probabilities must be in [0, 1]'

submission.to_csv('submission.csv', index=False)
print('Saved submission.csv')
submission.head()

## Next Steps / Ideas to Improve the Score

- Try `LightGBM` / `XGBoost` / `CatBoost` if not already installed (`pip install lightgbm xgboost`) &mdash; gradient boosting tends to do very well on synthetic Playground datasets.
- Blend/ensemble predictions from several strong models (simple average of `predict_proba` outputs, or a `StackingClassifier`).
- Since Playground datasets are synthetically generated, check the competition's **Discussion** and **Code** tabs &mdash; other competitors often find useful patterns in how the data was generated.
- Try target encoding for categorical features instead of one-hot, especially if any categorical column has high cardinality.
- Use `RandomizedSearchCV` or Optuna for a wider/faster hyperparameter search if the grid above is too slow.
- Consider combining the original real-world dataset the synthetic data was based on (if the competition description links to one) as extra training data.